# LangChain Agent with a Self-Hosted LLM on Azure Container Apps

This notebook demonstrates how to build an **AI agent** using **LangChain** and **LangGraph** that connects to a self-hosted Gemma 4 model running on Azure Container Apps (ACA) with vLLM.

Since vLLM exposes an **OpenAI-compatible API**, we use LangChain's `ChatOpenAI` class to connect — no Azure OpenAI resource required.

## Key Concepts

1. **ChatOpenAI** — LangChain's chat model wrapper, pointed at our custom vLLM endpoint.
2. **Tools** — Python functions decorated with `@tool` that the agent can invoke.
3. **ReAct Agent** — A LangGraph prebuilt agent that reasons, calls tools, and synthesizes answers.

In [1]:
%pip install langchain langgraph langchain-openai langchain-mcp-adapters

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Get the LLM Endpoint

Retrieve the FQDN of the Gemma 4 model deployed on ACA from the Terraform output.

In [1]:
aca_gemma4_31b_it_a100_fqdn = ! terraform output -raw aca_gemma4_31b_it_a100_fqdn
aca_gemma4_31b_it_a100_fqdn = aca_gemma4_31b_it_a100_fqdn.n
print("LLM Endpoint:", aca_gemma4_31b_it_a100_fqdn)

LLM Endpoint: gemma-4-31b-it-a100.gentlemushroom-793350b5.swedencentral.azurecontainerapps.io


## 1. Simple Chat — No Tools

Create a `ChatOpenAI` model pointing at the vLLM OpenAI-compatible endpoint and send a basic message. Note how we can enable streaming responses for real-time output.

In [48]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

model = ChatOpenAI(
    base_url=f"http://{aca_gemma4_31b_it_a100_fqdn}/v1",
    api_key="EMPTY",
    model="google/gemma-4-31B-it",
    streaming=True
)

for chunk in model.stream([HumanMessage(content="What is Azure Container Apps?")]):
    print(chunk.content, end="", flush=True)

**Azure Container Apps (ACA)** is a fully managed, serverless container service provided by Microsoft Azure. It is designed to allow developers to build and deploy scalable applications and microservices without having to manage the underlying infrastructure (like virtual machines, nodes, or complex Kubernetes clusters).

Think of it as a "middle ground" between **Azure App Service** (simple web apps) and **Azure Kubernetes Service (AKS)** (full orchestration power).

Here is a detailed breakdown of what makes Azure Container Apps unique:

---

### 1. The Core Concept: "Serverless Kubernetes"
Under the hood, Azure Container Apps is built on **Kubernetes**, **KEDA** (Kubernetes-based Event-Driven Autoscaling), and **Dapr** (Distributed Application Runtime). 

However, **you never see the Kubernetes cluster.** You don't manage nodes, upgrade versions, or configure pods manually. Azure handles all the "plumbing," allowing you to focus entirely on your code and your container image.

### 2

## 2. Define Tools

Create custom Python functions as tools using the `@tool` decorator. These will be available for the agent to call when needed.

In [26]:
from langchain_core.tools import tool
from random import randint


@tool
def get_weather(location: str) -> str:
    """Get the current weather for a given location."""
    conditions = ["sunny", "cloudy", "rainy", "stormy"]
    temp = randint(10, 30)
    return f"The weather in {location} is {conditions[randint(0, 3)]} with a high of {temp}°C."


@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers together.

    Args:
        a: first number
        b: second number
    """
    return a * b


tools = [get_weather, multiply]
print("Registered tools:", [t.name for t in tools])

Registered tools: ['get_weather', 'multiply']


## 3. Tool Binding — Test Tool Calling

Bind the tools to the model and verify the LLM can decide when to call them.

In [27]:
model_with_tools = model.bind_tools(tools)

# This should NOT trigger a tool call
response = model_with_tools.invoke([HumanMessage(content="Hi there!")])
print(f"Content: {response.content}")
print(f"Tool calls: {response.tool_calls}")

Content: Hello! How can I help you today?
Tool calls: []


In [28]:
# This SHOULD trigger a tool call
response = model_with_tools.invoke([HumanMessage(content="What's the weather in Amsterdam?")])
print(f"Content: {response.content}")
print(f"Tool calls: {response.tool_calls}")

Content: 
Tool calls: [{'name': 'get_weather', 'args': {'location': 'Amsterdam'}, 'id': 'chatcmpl-tool-9dff88b98272dfbc', 'type': 'tool_call'}]


## 4. Create a ReAct Agent

Use LangGraph's `create_react_agent` to build an agent that can reason about when to call tools, execute them, and incorporate results into its response.

The agent implements the **ReAct** (Reasoning + Acting) pattern: it thinks about what to do, calls a tool if needed, observes the result, and repeats until it has a final answer.

In [29]:
from langchain.agents import create_agent

agent = create_agent(model, tools)

## 5. Run the Agent

### No tool needed — simple question

In [30]:
response = agent.invoke({"messages": [HumanMessage(content="Hi! What is Kubernetes?")]})

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Hi! What is Kubernetes?
================================== Ai Message ==================================

**Kubernetes** (often abbreviated as **K8s**) is an open-source platform designed to automate the deployment, scaling, and management of **containerized applications**.

To understand Kubernetes, it helps to first understand containers.

### 1. The Background: What are Containers?
In the past, developers struggled with the "it works on my machine" problem—where code works on a laptop but fails on a server because of different settings or versions. 

**Containers** (like Docker) solve this by bundling the application code together with all the libraries, dependencies, and configuration files it needs to run. This makes the app portable and consistent across any environment.

### 2. Why do you need Kubernetes?
If you have one or two containers, you can manage them manually. But imagine a giant company l

### Tool call — weather query

The agent should recognize this requires the `get_weather` tool, call it, then respond with the result.

In [31]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What's the weather like in Amsterdam?")]}
)

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

What's the weather like in Amsterdam?
================================== Ai Message ==================================
Tool Calls:
  get_weather (chatcmpl-tool-90506cd11644dec7)
 Call ID: chatcmpl-tool-90506cd11644dec7
  Args:
    location: Amsterdam
================================= Tool Message =================================
Name: get_weather

The weather in Amsterdam is stormy with a high of 30°C.
================================== Ai Message ==================================

The weather in Amsterdam is currently stormy with a high of 30°C.


### Tool call — multiply

In [32]:
response = agent.invoke(
    {"messages": [HumanMessage(content="What is 7 multiplied by 13?")]}
)

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

What is 7 multiplied by 13?
================================== Ai Message ==================================
Tool Calls:
  multiply (chatcmpl-tool-8d8fa08a336aaacd)
 Call ID: chatcmpl-tool-8d8fa08a336aaacd
  Args:
    a: 7
    b: 13
================================= Tool Message =================================
Name: multiply

91
================================== Ai Message ==================================

7 multiplied by 13 is 91.


## 6. Streaming

Stream the agent's step-by-step reasoning and responses in real time. Each step (LLM thinking, tool call, tool result, final answer) is printed as it occurs.

In [34]:
for step in agent.stream(
    {"messages": [HumanMessage(content="What's the weather in Paris?")]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What's the weather in Paris?
================================== Ai Message ==================================
Tool Calls:
  get_weather (chatcmpl-tool-974de8562b972802)
 Call ID: chatcmpl-tool-974de8562b972802
  Args:
    location: Paris
================================= Tool Message =================================
Name: get_weather

The weather in Paris is stormy with a high of 26°C.
================================== Ai Message ==================================

The weather in Paris is currently stormy with a high of 26°C.


## 7. Microsoft Learn MCP Server

The [Model Context Protocol (MCP)](https://modelcontextprotocol.io/) lets you expose tools via a standard protocol that any MCP-compatible client can consume. This is especially powerful for **remote hosted MCP servers** — tools running as HTTP services that your agent can call over the network.

Microsoft exposes a public MCP server at `https://learn.microsoft.com/api/mcp` that provides tools to search and retrieve Microsoft documentation.

This is a great example of a **third-party hosted MCP server** — you don't need to deploy anything, just point your client at the URL.

In [35]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent

# Connect to the Microsoft Learn MCP server
learn_client = MultiServerMCPClient(
    {
        "microsoft-learn": {
            "url": "https://learn.microsoft.com/api/mcp",
            "transport": "http",
        }
    }
)

# Discover tools exposed by Microsoft Learn
learn_tools = await learn_client.get_tools()
print("Microsoft Learn MCP tools:", [t.name for t in learn_tools])

# Create an agent with the self-hosted LLM + Microsoft Learn tools
learn_agent = create_agent(model, learn_tools)

Microsoft Learn MCP tools: ['microsoft_docs_search', 'microsoft_code_sample_search', 'microsoft_docs_fetch']


### Query Microsoft Documentation

Ask the agent a question that requires looking up Microsoft documentation. The agent will call the Microsoft Learn MCP tools to search and retrieve relevant docs.

In [36]:
from langchain_core.messages import HumanMessage

response = await learn_agent.ainvoke(
    {"messages": [HumanMessage(content="How do I deploy a container app with GPU support on Azure Container Apps?")]}
)

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

How do I deploy a container app with GPU support on Azure Container Apps?
================================== Ai Message ==================================
Tool Calls:
  microsoft_docs_search (chatcmpl-tool-88d7593e11dff955)
 Call ID: chatcmpl-tool-88d7593e11dff955
  Args:
    query: deploy Azure Container Apps with GPU support
================================= Tool Message =================================
Name: microsoft_docs_search

[{'type': 'text', 'text': '{"results":[{"title":"Tutorial: Generate images using serverless GPUs in Azure Container Apps (azure-cli)","content":"# Tutorial: Generate images using serverless GPUs in Azure Container Apps (azure-cli)\\nIn this article, you learn how to create a container app that uses [serverless GPUs](https://learn.microsoft.com/azure/container-apps/gpu-serverless-overview) to power an AI application.\\nWith serverless GPUs, you have direct access to GPU compu

## 8. Using a Remote MCP Server as Tools

In this section we connect to the **Open Web Search MCP server** deployed on Azure Container Apps (see `aca_mcp_server.tf`) using [`langchain-mcp-adapters`](https://github.com/langchain-ai/langchain-mcp-adapters). The adapter discovers the server's tools automatically and makes them available as LangChain tools.

In [37]:
aca_mcp_server_fqdn = ! terraform output -raw aca_mcp_server_fqdn
aca_mcp_server_fqdn = aca_mcp_server_fqdn.n
print("MCP Server Endpoint:", aca_mcp_server_fqdn)

MCP Server Endpoint: mcp-server.gentlemushroom-793350b5.swedencentral.azurecontainerapps.io


### Connect to the Remote MCP Server and Discover Tools

Use `MultiServerMCPClient` to connect to the MCP server over **Streamable HTTP** transport. The client automatically discovers all tools the server exposes.

In [38]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent

# Connect to the remote MCP server over Streamable HTTP
client = MultiServerMCPClient(
    {
        "web-search": {
            "url": f"http://{aca_mcp_server_fqdn}/mcp",
            "transport": "http",
        }
    }
)

# Discover tools exposed by the MCP server
mcp_tools = await client.get_tools()
print("MCP tools discovered:", [t.name for t in mcp_tools])

# Create an agent with the self-hosted LLM + MCP tools
mcp_agent = create_agent(model, mcp_tools)

MCP tools discovered: ['search', 'fetchLinuxDoArticle', 'fetchCsdnArticle', 'fetchGithubReadme', 'fetchWebContent', 'fetchJuejinArticle']


### Run the Agent with MCP Tools

The agent will use the remote web search MCP tool to answer questions that require live information from the internet.

In [39]:
from langchain_core.messages import HumanMessage

response = await mcp_agent.ainvoke(
    {"messages": [HumanMessage(content="Search the web for the latest news about Azure Container Apps")]}
)

for msg in response["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Search the web for the latest news about Azure Container Apps
================================== Ai Message ==================================
Tool Calls:
  search (chatcmpl-tool-af8082ab9f2dec4a)
 Call ID: chatcmpl-tool-af8082ab9f2dec4a
  Args:
    query: latest news Azure Container Apps 2024 2025
================================= Tool Message =================================
Name: search

[{'type': 'text', 'text': '{\n  "query": "latest news Azure Container Apps 2024 2025",\n  "engines": [\n    "duckduckgo"\n  ],\n  "totalResults": 10,\n  "results": [\n    {\n      "title": "What&#x27;s new in Azure Container Apps at Ignite&#x27;25",\n      "url": "https://techcommunity.microsoft.com/blog/appsonazureblog/whats-new-in-azure-container-apps-at-ignite25/4470391",\n      "description": "<b>Azure</b> <b>Container</b> <b>Apps</b> (ACA) is a fully managed serverless <b>container</b> platform that enables devel

### Streaming with MCP Tools

Stream the agent's step-by-step reasoning as it decides to call the remote MCP web search tool and synthesizes the results.

In [41]:
async for step in mcp_agent.astream(
    {"messages": [HumanMessage(content="What is the current price of Bitcoin?")]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What is the current price of Bitcoin?
================================== Ai Message ==================================
Tool Calls:
  search (chatcmpl-tool-b437b1212e751112)
 Call ID: chatcmpl-tool-b437b1212e751112
  Args:
    query: current price of Bitcoin
================================= Tool Message =================================
Name: search

[{'type': 'text', 'text': '{\n  "query": "current price of Bitcoin",\n  "engines": [\n    "duckduckgo"\n  ],\n  "totalResults": 10,\n  "results": [\n    {\n      "title": "Bitcoin price today, BTC to USD live price, marketcap and chart ...",\n      "url": "https://coinmarketcap.com/currencies/bitcoin/",\n      "description": "The live <b>Bitcoin</b> <b>price</b> today is $68,153.75 USD with a 24-hour trading volume of $34,210,076,040.38 USD. We update our BTC to USD <b>price</b> in real-time.",\n      "source": "coinmarketcap.com",\n      "engine": "duckduckg

## More Resources

- [LangChain Tool Calling](https://python.langchain.com/docs/concepts/tool_calling/)
- [LangGraph ReAct Agent](https://python.langchain.com/docs/tutorials/agents/)
- [ChatOpenAI with custom endpoints](https://python.langchain.com/api_reference/openai/chat_models/langchain_openai.chat_models.base.ChatOpenAI.html)
- [LangChain MCP Adapters](https://github.com/langchain-ai/langchain-mcp-adapters) — connect LangChain agents to MCP servers
- [Model Context Protocol (MCP)](https://modelcontextprotocol.io/introduction) — the open standard for tool interoperability
- [Microsoft Learn MCP Server](https://learn.microsoft.com/api/mcp) — search Microsoft documentation via MCP